# Обучение ИИ-сегментатора текста для манги v2
Этот ноутбук предназначен для запуска в **Google Colab (с подключенным T4 GPU)**.
**Исправление v2**: теперь используется PIL/Pillow с японским шрифтом Noto Sans CJK — модель обучится распознавать настоящие иероглифы кандзи, хирагану и катакану.
В конце обучения модель будет экспортирована в формат `segmenter.onnx` для MangaEditor.

In [ ]:
# 1. Установка необходимых библиотек и скачивание японского шрифта
!pip install segmentation-models-pytorch albumentations onnx onnxruntime
!apt-get install -y fonts-noto-cjk
import glob
# Ищем установленный японский шрифт
fonts = glob.glob('/usr/share/fonts/**/*.otf', recursive=True) + glob.glob('/usr/share/fonts/**/*.ttf', recursive=True)
jp_fonts = [f for f in fonts if 'Noto' in f and ('JP' in f or 'CJK' in f or 'japanese' in f.lower())]
print('Найденные японские шрифты:')
for f in jp_fonts:
    print(f)

In [ ]:
# 2. Импорт библиотек
import os
import glob
import random
import numpy as np
import cv2
import torch
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

In [ ]:
# 3. Загрузка шрифтов
# Ищем все доступные японские шрифты
all_fonts = glob.glob('/usr/share/fonts/**/*.otf', recursive=True) + glob.glob('/usr/share/fonts/**/*.ttf', recursive=True)
jp_font_paths = [f for f in all_fonts if any(k in f for k in ['Noto', 'CJK', 'japanese', 'JP'])]
if not jp_font_paths:
    jp_font_paths = all_fonts  # Используем все доступные если Japanese не найден
    print('Японские шрифты не найдены, используем все доступные')
else:
    print(f'Найдено {len(jp_font_paths)} японских шрифтов')

# Набор японских символов (кандзи, хирагана, катакана)
HIRAGANA = 'あいうえおかきくけこさしすせそたちつてとなにぬねのはひふへほまみむめもやゆよらりるれろわをん'
KATAKANA = 'アイウエオカキクケコサシスセソタチツテトナニヌネノハヒフヘホマミムメモヤユヨラリルレロワヲン'
KANJI = '一二三四五六七八九十百千万年月日時人口大小上下左右中心前後東西南北国家言語食飲水火山川海空'
SYMBOLS = '！？…・「」『』【】'
LATIN = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789'
ALL_CHARS = HIRAGANA + KATAKANA + KANJI + SYMBOLS + LATIN
print(f'Итого символов в датасете: {len(ALL_CHARS)}')

In [ ]:
# 4. Генератор синтетических данных с PIL (умеет рисовать японский текст!)
class SyntheticMangaDataset(Dataset):
    def __init__(self, size=5000, transform=None):
        self.size = size
        self.transform = transform
        self.jp_font_paths = jp_font_paths
        self.all_chars = ALL_CHARS
        
    def __len__(self):
        return self.size
        
    def get_font(self, size):
        path = random.choice(self.jp_font_paths)
        try:
            return ImageFont.truetype(path, size)
        except:
            return ImageFont.load_default()
            
    def generate_screentone(self, width=256, height=256):
        bg = np.full((height, width), random.randint(215, 255), dtype=np.uint8)
        if random.random() > 0.35:
            dot_dist = random.randint(4, 14)
            dot_size = random.randint(1, 3)
            dot_color = random.randint(60, 190)
            ox, oy = random.randint(0, dot_dist), random.randint(0, dot_dist)
            for y in range(oy, height, dot_dist):
                for x in range(ox, width, dot_dist):
                    cv2.circle(bg, (x, y), dot_size, dot_color, -1)
        # Случайный шум
        if random.random() > 0.7:
            noise = np.random.randint(-10, 10, (height, width), dtype=np.int16)
            bg = np.clip(bg.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        return bg
        
    def draw_random_lines(self, img_np):
        num_lines = random.randint(0, 6)
        for _ in range(num_lines):
            p1 = (random.randint(0, 256), random.randint(0, 256))
            p2 = (random.randint(0, 256), random.randint(0, 256))
            cv2.line(img_np, p1, p2, random.randint(0, 120), random.randint(1, 4))
            
    def __getitem__(self, idx):
        # 1. Фон + линии рисунка
        bg_np = self.generate_screentone(256, 256)
        self.draw_random_lines(bg_np)
        
        # 2. Конвертируем в PIL для отрисовки японского текста
        img_pil = Image.fromarray(bg_np).convert('L')
        mask_pil = Image.new('L', (256, 256), 0)
        
        draw_img = ImageDraw.Draw(img_pil)
        draw_mask = ImageDraw.Draw(mask_pil)
        
        # 3. Рисуем 1-4 блока текста
        num_texts = random.randint(1, 4)
        for _ in range(num_texts):
            text_len = random.randint(2, 8)
            # Чаще японский текст (70%), иногда латинский (30%)
            if random.random() < 0.7:
                chars = HIRAGANA + KATAKANA + KANJI
                # Иногда вертикальный текст (с переносами)
                text = '\n'.join(random.choice(chars) for _ in range(text_len))
            else:
                text = ''.join(random.choice(LATIN) for _ in range(text_len))
            
            font_size = random.randint(14, 38)
            font = self.get_font(font_size)
            outline_w = random.randint(1, 4)
            
            # Позиция
            tx = random.randint(5, 200)
            ty = random.randint(5, 200)
            
            # Белая обводка
            outline_color = random.randint(210, 255)
            for dx in range(-outline_w, outline_w+1):
                for dy in range(-outline_w, outline_w+1):
                    if dx != 0 or dy != 0:
                        draw_img.text((tx+dx, ty+dy), text, font=font, fill=outline_color)
            
            # Черный текст
            text_color = random.randint(0, 30)
            draw_img.text((tx, ty), text, font=font, fill=text_color)
            
            # Маска (текст + обводка)
            mask_outline = outline_w + 1
            for dx in range(-mask_outline, mask_outline+1):
                for dy in range(-mask_outline, mask_outline+1):
                    draw_mask.text((tx+dx, ty+dy), text, font=font, fill=255)
        
        # 4. Конвертируем назад в numpy
        img = np.array(img_pil).astype(np.float32) / 255.0
        mask = (np.array(mask_pil) > 0).astype(np.float32)
        
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            img = torch.tensor(img).unsqueeze(0)
            mask = torch.tensor(mask).unsqueeze(0)
            
        return img, mask

In [ ]:
# 5. Визуальная проверка: смотрим пример из датасета
import matplotlib.pyplot as plt
sample_ds = SyntheticMangaDataset(size=4)
img, mask = sample_ds[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img.squeeze().numpy(), cmap='gray')
axes[0].set_title('Входная картинка (манга + текст)')
axes[1].imshow(mask.squeeze().numpy(), cmap='gray')
axes[1].set_title('Целевая маска (только текст)')
plt.tight_layout()
plt.savefig('sample_check.png')
plt.show()
print('Если на маске видны буквы/иероглифы — датасет работает правильно!')

In [ ]:
# 6. Аугментации и DataLoader
train_transform = A.Compose([
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=10, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
    A.GaussNoise(var_limit=(0.001, 0.008), p=0.3),
    ToTensorV2()
])

val_transform = A.Compose([ToTensorV2()])

train_dataset = SyntheticMangaDataset(size=5000, transform=train_transform)
val_dataset = SyntheticMangaDataset(size=500, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
# 7. Определение модели U-Net с ResNet34
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используем устройство: {device}')

model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=1,
    classes=1,
    activation=None
)
model = model.to(device)

In [ ]:
# 8. Функция потерь и оптимизатор
criterion = smp.losses.DiceLoss(mode='binary', from_logits=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300)

In [ ]:
# 9. Тренировочный цикл — 300 эпох
num_epochs = 300
best_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    
    for imgs, masks in tqdm(train_loader, desc=f'Эпоха {epoch+1}/{num_epochs}'):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_loader.dataset)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            val_loss += loss.item() * imgs.size(0)
    val_loss /= len(val_loader.dataset)
    
    scheduler.step()
    print(f'Эпоха {epoch+1} завершена. Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
    
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), 'best_manga_segmenter.pth')
        print('-> Сохранены новые лучшие веса!')

In [ ]:
# 10. Экспорт в ONNX для MangaEditor
model.load_state_dict(torch.load('best_manga_segmenter.pth', map_location='cpu'))
model.eval()

dummy_input = torch.randn(1, 1, 256, 256, dtype=torch.float32)
onnx_path = 'segmenter.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f'Модель успешно экспортирована в {onnx_path}!')
print('Скачайте оба файла: segmenter.onnx и segmenter.onnx.data (если он появился)')
print('и положите их в папку models вашего MangaEditor!')

# Скачиваем файл сразу из Colab
from google.colab import files
files.download('segmenter.onnx')
import os
if os.path.exists('segmenter.onnx.data'):
    files.download('segmenter.onnx.data')